# Regularisierung

Beim Trainieren und Evaluieren eines Modells müssen Sie sich oft mit dem Problem des Over- oder Underfittings auseinandersetzen. Eine einfache Möglichkeit, Overfitting eines Polynommodells zu verhindern, besteht darin, die Anzahl und Grade der Polynome zu reduzieren. Das in Scikit-Learn implementierte lineare Regressionsmodell bietet keinen weiteren Hyperparameter zur Einschränkung der Komplexität des Modells. Glücklicherweise bietet Scikit-Learn zwei andere lineare Modelle: `Ridge Regression` und `Lasso Regression`. Beide Modelle können verwendet werden, um die Modellkomplexität zu reduzieren und Overfitting zu verhindern, das bei einfacher linearer Regression auftreten kann.


In diesem Notebook lernen Sie:
* warum Regularisierung verwendet wird
* wie Ridge-Regression angewendet wird
* wie Lasso-Regression angewendet wird
* wie Ridge- und Lasso-Regularisierung die Parameter/Koeffizienten des Modells etwas unterschiedlich einschränken
* welcher Hyperparameter für Ridge- und Lasso-Regression angepasst werden kann, um die Modellkomplexität zu beeinflussen

## Daten



Wir werden den **Wine Quality Dataset** verwenden, um zu zeigen, wie Ridge- und Lasso-Regression die Modellkomplexität einschränken. Unser Datensatz hat 11 Variablen, um die Wein-*Qualität* vorherzusagen. 
Für den Zweck dieser Übung werden wir den Datensatz auch erweitern, indem wir Interaktionsmerkmale generieren. Das bedeutet, wir werden nicht nur die ursprünglichen Merkmale verwenden, sondern auch alle Produkte von Merkmalspaaren. 

Die Variablen sind (basierend auf physikalisch-chemischen Tests): 

1 - Feste Säure (fixed acidity)

2 - Flüchtige Säure (volatile acidity)

3 - Zitronensäure (citric acid)

4 - Restzucker (residual sugar)

5 - Chloride (chlorides)

6 - Freies Schwefeldioxid (free sulfur dioxide)

7 - Gesamtes Schwefeldioxid (total sulfur dioxide)

8 - Dichte (density)

9 - pH-Wert (pH)

10 - Sulfate (sulphates)

11 - Alkohol (alcohol)

Ausgabevariable (basierend auf sensorischen Daten):

12 - Qualität (quality) (Bewertung zwischen 0 und 10)


## Imports und Daten

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


# Define random seed
RSEED = 0

# define figure size
plt.rcParams['figure.figsize'] = (10, 7)

In [ ]:
df = pd.read_csv('data/winequality-white.csv',sep=";")
df.head(2)

In [ ]:
df.info()

Alle unsere Merkmale sind numerisch und es gibt keine fehlenden Werte. Um zu verstehen, wie verschiedene Regularisierungsmethoden funktionieren, können wir die Datenbereinigung vorerst überspringen.

In [ ]:
# define features and target
features = df.columns.tolist()
features.remove('quality')
X = df[features]
y = df['quality']

In [ ]:
#correlation plot
ax = sns.heatmap(df.corr(), annot=True)

## Feature Engineering


Wir werden Interaktionsterme (= Merkmalskombinationen, z.B. X1*X2) aller Variablen erstellen, die im Datensatz existieren.
Wenn wir also drei Merkmale (X1, X2, X3) haben und alle Interaktionsmerkmale vom Grad 2 (nur Kombinationen von zwei!) erstellen möchten, erhalten wir insgesamt 6 Merkmale. Die drei Startmerkmale plus alle Kombinationen von 2 Merkmalen: (X1, X2, X3, X1\*X2, X1\*X3, X2\*X3). 
Wenn Sie alle Feature-Engineering-Möglichkeiten der `PolynomialFeatures`-Methode verstehen möchten, [schauen Sie sich die Dokumentation von scikit-learn an](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html).

In [ ]:
# generate interaction features
poly = PolynomialFeatures(interaction_only=True, include_bias=False, degree=2)
X_poly = poly.fit_transform(X)


Um zu berechnen, wie viele Interaktionsterme nur für unsere 11 Merkmale (n) und Grad 2 (k) (nur Kombinationen von 2 Merkmalen) existieren, können Sie diese Formel zur Berechnung von [Kombinationen](https://en.wikipedia.org/wiki/Combination) verwenden: 

$\displaystyle {\binom {n}{k}}={\frac {n(n-1)\dotsb (n-k+1)}{k(k-1)\dotsb 1}}=  {\frac {n!}{k!(n-k)!}}$

In [ ]:
# In our case with 11 features, we end up with 66 features (11 features + 55 combinations)
poly.n_output_features_

## Train-Test-Split


Bevor wir unsere Modelle trainieren, teilen wir die Daten auf, um einen Testdatensatz für die Evaluierung unserer Modelle zu haben.

In [ ]:
# train-test-split
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, random_state=RSEED, test_size=0.4)


In [ ]:
# Let's check the shape of our dataframes
print("Training samples: ", X_train.shape[0],"  Nr. of Features: ", X_train.shape[1])
print("Training lables: ", y_train.shape[0])
print( "----"*12)
print("Test samples: ",X_test.shape[0],"  Nr. of Features: ",X_test.shape[1])
print("Test lables: ",y_test.shape[0])

## Lineare Regression

Wir beginnen mit dem Training eines linearen Regressionsmodells. 
Wir können sklearns `LinearRegression`-Klasse verwenden, um ein Modell zu trainieren, das alle unsere verfügbaren Merkmale enthält.

Für jedes unserer Modelle werden wir die Leistung bewerten, indem wir uns den RMSE (Root Mean Squared Error) ansehen.



In [ ]:
from sklearn.linear_model import LinearRegression

# initialize and train model
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# predict on test-set
y_pred_lin = lin_reg.predict(X_test)
y_pred_train = lin_reg.predict(X_train)

# RMSE on train and test set
print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 3))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_lin)), 3))

Wir können sehen, dass unser Modell gut auf den Trainingsdaten funktioniert, aber der RMSE auf dem Testsatz ist viel schlechter. Diese Diskrepanz zwischen der Leistung auf den Trainings- und Testsätzen ist ein klares Zeichen für Overfitting. Da `LinearRegression` keine Hyperparameter zur Einschränkung der Modellkomplexität hat, müssen wir ein anderes Modell verwenden. 

Zum Vergleich werden wir uns auch den Wert des höchsten Koeffizienten ansehen und wie viele Koeffizienten null sind.

In [ ]:
def coeff_info(model):
    coeff_used = np.sum(model.coef_!=0)
    print('The model is using', coeff_used, 'out of 66 features.')
    print( "The highest coefficient has a value of:", max(model.coef_.round(3)))

In [ ]:
coeff_info(lin_reg)

## Ridge-Regression (L2-Regularisierung)



Eine häufig verwendete Alternative zum "normalen" linearen Regressionsmodell ist die `Ridge`-Regression. Sie ist auch ein lineares Modell, das im Grunde dieselbe Formel verwendet, die für gewöhnliche kleinste Quadrate verwendet wird. Unser Ridge-Regressionsmodell wird jedoch auch versuchen, die Größe der Koeffizienten so klein wie möglich zu halten. Mit anderen Worten, alle Einträge von *w* sollten nahe bei null liegen. Wir können auch sagen, jedes Merkmal sollte so wenig Einfluss wie möglich auf das Ergebnis haben, während es dennoch gut vorhersagt. 

Wenn Sie einen Blick in die [Dokumentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) werfen, können Sie sehen, dass dieses Modell einen Hyperparameter namens *alpha* hat, der angepasst werden kann. Die Erhöhung von Alpha zwingt die Koeffizienten, sich mehr in Richtung null zu bewegen, was die Trainingsleistung verringert, aber die Generalisierung verbessern kann. 

Schauen wir uns an, wie Ridge-Regression mit verschiedenen Alpha-Werten auf unserem Datensatz abschneidet:

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
# initialize and train model with (default value) alpha = 10
ridge_10 = Ridge(alpha=10)
ridge_10.fit(X_train, y_train)

# predict on test-set
y_pred_ridge = ridge_10.predict(X_test)
y_pred_train = ridge_10.predict(X_train)

# RMSE of test set
print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 3))
print("Test  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge)), 3))

In [ ]:
coeff_info(ridge_10)

In [ ]:
# initialize and train model with alpha = 1
ridge_1 = Ridge(alpha=1)
ridge_1.fit(X_train, y_train)

# predict on test-set
y_pred_ridge_1 = ridge_1.predict(X_test)
y_pred_train_1 = ridge_1.predict(X_train)

# RMSE of test set

print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train_1)), 3))
print("Test  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge_1)), 3))

In [ ]:
coeff_info(ridge_1)

In [ ]:
# initialize and train model with alpha = 0.1
ridge_01 = Ridge(alpha=0.1)
ridge_01.fit(X_train, y_train)

# predict on test-set
y_pred_ridge_01 = ridge_01.predict(X_test)
y_pred_train_01 = ridge_01.predict(X_train)

# RMSE of test set
print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train_01)), 3))
print("Test  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge_01)), 3))

In [ ]:
coeff_info(ridge_01)

Sie können sehen, dass alle drei Modelle deutlich besser abschneiden als das lineare Regressionsmodell und der Unterschied zwischen Trainings- und Test-Score ist erheblich geringer. Daher overfittet unser Modell nicht so stark wie die lineare Regression.  
In unserem Fall liefert der Wert alpha = 10 die besten Ergebnisse. Die optimale Einstellung von Alpha hängt vom jeweiligen Datensatz ab, den wir verwenden.


Wir können auch die Größe unserer Koeffizienten visualisieren. 
Auf der x-Achse sind die Nummern aller unserer Koeffizienten (wir haben 66 Variablen) als Koeffizientenindex (b1, b2, b3 etc.) aufgetragen und auf der y-Achse die Werte für diese Koeffizienten.


In [ ]:
# plot of different coefficients
# linear regression without regularization
plt.plot(lin_reg.coef_, 'o', alpha=0.5, label="LinearRegression", color='green')
# ridge regression with lowest alpha value = regularization term
plt.plot(ridge_01.coef_, 'v', label="Ridge alpha=0.1", color='orange')
# ridge regression with middle alpha value = regularization term
plt.plot(ridge_1.coef_, '*', label="Ridge alpha=1", color='red')
# ridge regression with highest alpha value = regularization term
plt.plot(ridge_10.coef_, 's', alpha=0.5, label="Ridge alpha=10", color='blue')


plt.xlabel("Coefficient index", fontsize=16)
plt.ylabel("Coefficient magnitude", fontsize=16)
plt.hlines(0, 0, len(lin_reg.coef_))
plt.ylim(-12, 10)
plt.legend(fontsize=13, ncol=2, loc=(0, 1.05));

Das Diagramm zeigt schön, wie die Koeffizienten umso näher an null kommen, je höher die Alpha-Werte werden. 
Die grünen Punkte, die die Koeffizienten unseres linearen Regressionsmodells ohne jegliche Regularisierung darstellen, sind manchmal so groß, dass sie fast außerhalb unseres Diagramms liegen.

## Lasso-Regression (L1-Regularisierung)



Eine Alternative zu `Ridge` ist die `Lasso`-Regression. Ähnlich wie bei der Ridge-Regression schränkt Lasso die Koeffizienten ein, nahe bei null zu sein. Es tut dies auf eine etwas andere Weise, sodass bei Verwendung von Lasso einige Koeffizienten genau null werden. Dies bedeutet, dass einige Merkmale vom Modell vollständig ignoriert werden. Es kann als automatische Merkmalsauswahl angesehen werden, die Modelle oft leichter interpretierbar macht und die wichtigsten Merkmale aufdecken kann.



Ein Blick in die [Dokumentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html?highlight=lasso#sklearn.linear_model.Lasso) zeigt uns, dass Lasso auch einen Hyperparameter namens *alpha* hat, der angepasst werden kann. Die Senkung von Alpha ermöglicht es uns, ein komplexeres Modell anzupassen. Wenn Alpha jedoch zu niedrig eingestellt ist, erhalten wir am Ende ein ähnliches Ergebnis wie bei der linearen Regression, da keine Regularisierung stattfindet.  

Lassen Sie uns Lasso-Regression mit verschiedenen Alpha-Werten ausprobieren:

In [ ]:
from sklearn.linear_model import Lasso

In [ ]:
# initialize and train model with (default value) alpha = 0.5
lasso_05 = Lasso(alpha=0.5, max_iter=int(10e5))
lasso_05.fit(X_train,y_train)

# predict on test-set
y_pred_lasso = lasso_05.predict(X_test)
y_pred_train_lasso = lasso_05.predict(X_train)

# RMSE of test set
print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train_lasso)), 3))
print("Test  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_lasso)), 3))


In [ ]:
coeff_info(lasso_05)

In [ ]:
# initialize and train model with alpha 0.05

lasso_005 = Lasso(alpha=0.05, max_iter=100000)
lasso_005.fit(X_train,y_train)

# predict on test-set
y_pred_lasso_005 = lasso_005.predict(X_test)
y_pred_train_lasso_005 = lasso_005.predict(X_train)

# RMSE of test set
print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train_lasso_005)), 3))
print("Test  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_lasso_005)), 3))


In [ ]:
coeff_info(lasso_005)

In [ ]:
# initialize and train model with alpha 0.005
lasso_0005 = Lasso(alpha=0.005, max_iter=100000)
lasso_0005.fit(X_train,y_train)

# predict on test-set
y_pred_lasso_0005 = lasso_0005.predict(X_test)
y_pred_train_lasso_0005 = lasso_0005.predict(X_train)

# RMSE of test set
print("Train RMSE:", round(np.sqrt(mean_squared_error(y_train, y_pred_train_lasso_0005)), 3))
print("Test  RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_lasso_0005)), 3))

In [ ]:
coeff_info(lasso_0005)

Sie können sehen, dass unser Lasso-Modell mit alpha = 0,5 ziemlich schlecht abschneidet und nur 10 Merkmale verwendet. Die Senkung von Alpha, um ein komplexeres Modell zu ermöglichen, führt zu einer besseren Leistung. Der RMSE unserer Modelle mit alpha = 0,05 und alpha = 0,005 ist ziemlich ähnlich, aber es gibt immer noch einen ziemlichen Unterschied in der Anzahl der verwendeten Merkmale. Da weniger Merkmale es oft einfacher machen, unser Modell zu interpretieren, entscheiden wir uns für alpha = 0,05. 

Wir können auch die Koeffizienten unserer Lasso-Modelle visualisieren.

In [ ]:
# plot of different coefficients
plt.plot(lin_reg.coef_, 'o', alpha=0.3, label="Linear Regression", color='green')
# lasso regression with lowest alpha value = regularization term
plt.plot(lasso_0005.coef_, 'v', alpha=0.6, label="Lasso alpha=0.005", color='orange')
# lasso regression with middle alpha value = regularization term
plt.plot(lasso_005.coef_, 's', alpha=0.3, label="Lasso alpha=0.05", color='blue')
# lasso regression with highest alpha value = regularization term
plt.plot(lasso_05.coef_, '*', alpha=0.3,label="Lasso alpha=0.5", color='red')

plt.xlabel("Coefficient index", fontsize=16)
plt.ylabel("Coefficient magnitude", fontsize=16)
plt.hlines(0, 0, len(lin_reg.coef_))
plt.ylim(-15, 15)
plt.legend(fontsize=13, ncol=2, loc=(0, 1.05));

Um die Unterschiede in den Koeffizientenwerten für Lasso-Regularisierung wirklich zu sehen, zoomen wir etwas mehr hinein. Denken Sie daran, dass die meisten Koeffizienten für lineare Regression nicht im gezeichneten y-Bereich liegen werden.

In [ ]:
# plot of different coefficients
plt.plot(lin_reg.coef_, 'o', alpha=0.3, label="Linear Regression", color='green')
# lasso regression with lowest alpha value = regularization term
plt.plot(lasso_0005.coef_, 'v', alpha=0.6, label="Lasso alpha=0.005", color='orange')
# lasso regression with middle alpha value = regularization term
plt.plot(lasso_005.coef_, 's', alpha=0.3, label="Lasso alpha=0.05", color='blue')
# lasso regression with highest alpha value = regularization term
plt.plot(lasso_05.coef_, '*', alpha=0.3,label="Lasso alpha=0.5", color='red')

plt.xlabel("Coefficient index", fontsize=16)
plt.ylabel("Coefficient magnitude", fontsize=16)
plt.hlines(0, 0, len(lin_reg.coef_))
plt.ylim(-1, 1)
plt.legend(fontsize=13, ncol=2, loc=(0, 1.05));

Für alpha = 0,5 können wir sehen, dass die meisten Koeffizienten tatsächlich null sind und die anderen auch ziemlich klein sind. Auch wenn das Diagramm für unsere Ridge-Regressionen ähnlich aussah, waren die Koeffizienten der Ridge-Modelle nahe, aber nie genau null.

## Zusammenfassung



Nun haben Sie zwei verschiedene Möglichkeiten gesehen, um zu verhindern, dass Ihr lineares Regressionsmodell overfittet. In der Praxis ist Ridge-Regression normalerweise die erste Wahl. Wenn Sie jedoch eine große Anzahl von Merkmalen haben und die Interpretierbarkeit Ihres Modells verbessern möchten, ist es sinnvoll, sich für Lasso-Regression zu entscheiden, da sie einige Ihrer Merkmale eliminiert. 

Es gibt aber auch ein drittes Modell, das in scikit-learn implementiert ist: die [`ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html?highlight=elasticnet#sklearn.linear_model.ElasticNet)-Regression. Dieses Modell kombiniert die Strafen von Lasso und Ridge. Oft funktioniert diese Kombination am besten.

Fühlen Sie sich frei, mit dem `ElasticNet`-Regressionsmodell zu experimentieren und die Ergebnisse mit den anderen Regularisierungsmethoden zu vergleichen.

### Weiterführende Literatur


Die Beispiele in diesem Notebook finden Sie ähnlich in Andreas C. Müllers und Sarah Guidos Buch **Introduction to Machine Learning with Python**. Wenn Sie weitere Erklärungen benötigen, werfen Sie einen Blick auf das Kapitel über überwachtes Lernen/lineare Regression.